# CIFAR-10 embeddings + MLP classifier for C++ inference

This notebook:

1. Downloads **CIFAR-10**
2. Computes a fixed embedding for each image using a pretrained **ResNet-18**
3. Trains an **MLP classifier** on top of those embeddings
4. Evaluates accuracy
5. Exports the trained MLP weights in a **C++-friendly binary format**

## Export format

For each dense layer:
- weights are stored as `float32`
- weight shape is **[out_features, in_features]**
- binary layout is **row-major**
- bias is stored as `float32` vector of shape **[out_features]**

This matches the C++ formula:

```cpp
y[o] = b[o] + sum_i x[i] * W[o, i];
```

## Notes

- The embedding model is only used offline in Python for preprocessing.
- The exported model is only the **MLP classifier**, which is what you can use in your C++/OpenMP/MPI project.
- By default, the encoder output dimension is **512** (ResNet-18 penultimate layer).


In [1]:
# If needed, install dependencies in your notebook environment:
# %pip install torch torchvision tqdm numpy

In [3]:
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets, models, transforms
from torchvision.models import ResNet18_Weights
from tqdm.auto import tqdm

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.11.0+cu130
CUDA available: False


In [5]:
# ----------------------------
# Configuration
# ----------------------------
CONFIG = {
    "seed": 42,
    "data_dir": "./data",
    "work_dir": "./cifar10_embedding_mlp",
    "batch_size_embedding": 256,
    "batch_size_train": 256,
    "num_workers": 2,
    "embedding_dtype": "float32",   # "float32" recommended for export compatibility
    "mlp_hidden_dims": [4096, 4096, 4096],  # large hidden layers for HPC inference
    "dropout": 0.1,
    "learning_rate": 1e-3,
    "weight_decay": 1e-4,
    "epochs": 10,
    "val_fraction": 0.1,
    "recompute_embeddings": False,
    "export_dir": "./cifar10_embedding_mlp/export",
}

Path(CONFIG["work_dir"]).mkdir(parents=True, exist_ok=True)
Path(CONFIG["export_dir"]).mkdir(parents=True, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

## Load CIFAR-10 and build embedding transform

The encoder uses the standard preprocessing associated with the pretrained ResNet-18 weights:
- resize to 224
- normalize with ImageNet mean/std

CIFAR-10 labels:
`airplane, automobile, bird, cat, deer, dog, frog, horse, ship, truck`


In [6]:
weights = ResNet18_Weights.DEFAULT
encoder_preprocess = weights.transforms()

train_base = datasets.CIFAR10(root=CONFIG["data_dir"], train=True, download=True)
test_base = datasets.CIFAR10(root=CONFIG["data_dir"], train=False, download=True)

class_names = train_base.classes
num_classes = len(class_names)
print("Classes:", class_names)
print("Train size:", len(train_base), "Test size:", len(test_base))

100.0%
/home/pshvaiko/miniconda3/envs/mlp-cifar/lib/python3.11/site-packages/torchvision/datasets/cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Classes: ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
Train size: 50000 Test size: 10000


In [7]:
class CIFAR10WithTransform(torch.utils.data.Dataset):
    def __init__(self, base_dataset, transform):
        self.base_dataset = base_dataset
        self.transform = transform

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, label = self.base_dataset[idx]
        image = self.transform(image)
        return image, label

train_embed_ds = CIFAR10WithTransform(train_base, encoder_preprocess)
test_embed_ds = CIFAR10WithTransform(test_base, encoder_preprocess)

train_embed_loader = DataLoader(
    train_embed_ds,
    batch_size=CONFIG["batch_size_embedding"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

test_embed_loader = DataLoader(
    test_embed_ds,
    batch_size=CONFIG["batch_size_embedding"],
    shuffle=False,
    num_workers=CONFIG["num_workers"],
    pin_memory=torch.cuda.is_available(),
)

## Build pretrained image encoder

We use **ResNet-18** and remove the final classification layer.
The resulting embedding dimension is **512**.


In [8]:
encoder = models.resnet18(weights=weights)
encoder.fc = nn.Identity()
encoder = encoder.to(device)
encoder.eval()

# Infer embedding dimension
with torch.no_grad():
    dummy = torch.randn(1, 3, 224, 224, device=device)
    emb_dim = encoder(dummy).shape[1]

print("Embedding dimension:", emb_dim)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /home/pshvaiko/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100.0%

Embedding dimension: 512


In [9]:
@torch.no_grad()
def compute_embeddings(encoder, loader, device):
    all_embeddings = []
    all_labels = []

    encoder.eval()
    for images, labels in tqdm(loader, desc="Embedding", leave=False):
        images = images.to(device, non_blocking=True)
        embeddings = encoder(images)

        if CONFIG["embedding_dtype"] == "float32":
            embeddings = embeddings.float()
        elif CONFIG["embedding_dtype"] == "float16":
            embeddings = embeddings.half()
        else:
            raise ValueError("Unsupported embedding_dtype")

        all_embeddings.append(embeddings.cpu())
        all_labels.append(labels)

    x = torch.cat(all_embeddings, dim=0)
    y = torch.cat(all_labels, dim=0)
    return x, y

train_emb_path = Path(CONFIG["work_dir"]) / "train_embeddings.pt"
test_emb_path = Path(CONFIG["work_dir"]) / "test_embeddings.pt"

if CONFIG["recompute_embeddings"] or not train_emb_path.exists() or not test_emb_path.exists():
    print("Computing embeddings...")
    train_x_all, train_y_all = compute_embeddings(encoder, train_embed_loader, device)
    test_x, test_y = compute_embeddings(encoder, test_embed_loader, device)
    torch.save({"x": train_x_all, "y": train_y_all}, train_emb_path)
    torch.save({"x": test_x, "y": test_y}, test_emb_path)
else:
    print("Loading cached embeddings...")
    train_blob = torch.load(train_emb_path, map_location="cpu")
    test_blob = torch.load(test_emb_path, map_location="cpu")
    train_x_all, train_y_all = train_blob["x"], train_blob["y"]
    test_x, test_y = test_blob["x"], test_blob["y"]

print("train_x_all:", tuple(train_x_all.shape), train_x_all.dtype)
print("test_x:", tuple(test_x.shape), test_x.dtype)

Computing embeddings...


Embedding:   0%|          | 0/196 [00:00<?, ?it/s]

Embedding:   0%|          | 0/40 [00:00<?, ?it/s]

train_x_all: (50000, 512) torch.float32
test_x: (10000, 512) torch.float32


In [10]:
# Split training embeddings into train / validation
n_total = train_x_all.shape[0]
n_val = int(n_total * CONFIG["val_fraction"])
n_train = n_total - n_val

perm = torch.randperm(n_total)
train_idx = perm[:n_train]
val_idx = perm[n_train:]

train_x = train_x_all[train_idx].float()
train_y = train_y_all[train_idx].long()
val_x = train_x_all[val_idx].float()
val_y = train_y_all[val_idx].long()
test_x = test_x.float()
test_y = test_y.long()

print("Train:", train_x.shape, train_y.shape)
print("Val:  ", val_x.shape, val_y.shape)
print("Test: ", test_x.shape, test_y.shape)

Train: torch.Size([45000, 512]) torch.Size([45000])
Val:   torch.Size([5000, 512]) torch.Size([5000])
Test:  torch.Size([10000, 512]) torch.Size([10000])


## Train MLP classifier on top of embeddings

This is the model you will export and run in C++.


In [11]:
class MLPClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dim, dropout=0.0):
        super().__init__()
        layers = []
        dims = [input_dim] + list(hidden_dims)

        for in_dim, out_dim in zip(dims[:-1], dims[1:]):
            layers.append(nn.Linear(in_dim, out_dim))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(dims[-1], output_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

mlp = MLPClassifier(
    input_dim=emb_dim,
    hidden_dims=CONFIG["mlp_hidden_dims"],
    output_dim=num_classes,
    dropout=CONFIG["dropout"],
).to(device)

total_params = sum(p.numel() for p in mlp.parameters())
print(mlp)
print(f"Total parameters: {total_params:,}")

MLPClassifier(
  (net): Sequential(
    (0): Linear(in_features=512, out_features=4096, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=4096, out_features=4096, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=4096, out_features=4096, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.1, inplace=False)
    (9): Linear(in_features=4096, out_features=10, bias=True)
  )
)
Total parameters: 35,704,842


In [12]:
train_loader = DataLoader(
    TensorDataset(train_x, train_y),
    batch_size=CONFIG["batch_size_train"],
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    TensorDataset(val_x, val_y),
    batch_size=CONFIG["batch_size_train"],
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    TensorDataset(test_x, test_y),
    batch_size=CONFIG["batch_size_train"],
    shuffle=False,
    num_workers=0,
)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    mlp.parameters(),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

In [13]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == yb).sum().item()
        total_count += xb.size(0)

    return total_loss / total_count, total_correct / total_count


def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    for xb, yb in tqdm(loader, desc="Train", leave=False):
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * xb.size(0)
        preds = torch.argmax(logits, dim=1)
        total_correct += (preds == yb).sum().item()
        total_count += xb.size(0)

    return total_loss / total_count, total_correct / total_count

In [14]:
best_val_acc = -1.0
best_path = Path(CONFIG["work_dir"]) / "best_mlp.pt"
history = []

for epoch in range(1, CONFIG["epochs"] + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(mlp, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(mlp, val_loader, criterion, device)

    epoch_time = time.time() - t0
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "epoch_time_sec": epoch_time,
    })

    print(
        f"Epoch {epoch:2d} | "
        f"train loss {train_loss:.4f} | train acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} | val acc {val_acc:.4f} | "
        f"time {epoch_time:.1f}s"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            "model_state_dict": mlp.state_dict(),
            "config": CONFIG,
            "embedding_dim": emb_dim,
            "num_classes": num_classes,
            "class_names": class_names,
        }, best_path)

print("Best val acc:", best_val_acc)

Train:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch  1 | train loss 0.6627 | train acc 0.7767 | val loss 0.4645 | val acc 0.8360 | time 103.5s


Train:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch  2 | train loss 0.4260 | train acc 0.8552 | val loss 0.4228 | val acc 0.8482 | time 104.7s


Train:   0%|          | 0/176 [00:00<?, ?it/s]

...

Epoch  8 | train loss 0.2474 | train acc 0.9123 | val loss 0.4128 | val acc 0.8658 | time 92.5s


Train:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch  9 | train loss 0.2246 | train acc 0.9206 | val loss 0.3973 | val acc 0.8700 | time 91.4s


Train:   0%|          | 0/176 [00:00<?, ?it/s]

Epoch 10 | train loss 0.2042 | train acc 0.9264 | val loss 0.4064 | val acc 0.8738 | time 91.5s
Best val acc: 0.874


In [15]:
checkpoint = torch.load(best_path, map_location=device)
mlp.load_state_dict(checkpoint["model_state_dict"])

val_loss, val_acc = evaluate(mlp, val_loader, criterion, device)
test_loss, test_acc = evaluate(mlp, test_loader, criterion, device)

print(f"Best checkpoint | val loss {val_loss:.4f} | val acc {val_acc:.4f}")
print(f"Best checkpoint | test loss {test_loss:.4f} | test acc {test_acc:.4f}")

Best checkpoint | val loss 0.3813 | val acc 0.8740
Best checkpoint | test loss 0.4126 | test acc 0.8673


## Export MLP for C++

This exports **only the MLP classifier**, not the ResNet encoder.


In [16]:
def export_mlp_to_cpp(model: nn.Module, export_dir: str, input_dim: int, class_names):
    export_dir = Path(export_dir)
    export_dir.mkdir(parents=True, exist_ok=True)

    linear_layers = [m for m in model.modules() if isinstance(m, nn.Linear)]
    metadata = {
        "format": "mlp_classifier_v1",
        "dtype": "float32",
        "input_dim": input_dim,
        "num_classes": len(class_names),
        "class_names": list(class_names),
        "layers": [],
    }

    for layer_idx, layer in enumerate(linear_layers):
        weight = layer.weight.detach().cpu().float().contiguous().numpy()  # [out, in]
        bias = layer.bias.detach().cpu().float().contiguous().numpy()      # [out]

        weight_path = export_dir / f"layer_{layer_idx:02d}_weight.bin"
        bias_path = export_dir / f"layer_{layer_idx:02d}_bias.bin"

        weight.tofile(weight_path)
        bias.tofile(bias_path)

        metadata["layers"].append({
            "layer_index": layer_idx,
            "type": "linear",
            "in_features": int(layer.in_features),
            "out_features": int(layer.out_features),
            "weight_file": weight_path.name,
            "bias_file": bias_path.name,
        })

    with open(export_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print(f"Exported {len(linear_layers)} linear layers to: {export_dir}")


# Save best model in eval mode
mlp.eval()
export_mlp_to_cpp(mlp, CONFIG["export_dir"], emb_dim, class_names)

Exported 4 linear layers to: cifar10_embedding_mlp/export


In [17]:
# Export a small correctness-check sample for your C++ program
sample_count = 16
sample_x = test_x[:sample_count].contiguous().float()
with torch.no_grad():
    sample_logits = mlp(sample_x.to(device)).cpu().contiguous().float()
    sample_pred = torch.argmax(sample_logits, dim=1)

sample_x.numpy().tofile(Path(CONFIG["export_dir"]) / "test_input_embeddings.bin")
sample_logits.numpy().tofile(Path(CONFIG["export_dir"]) / "test_output_logits.bin")
sample_pred.numpy().astype(np.int64).tofile(Path(CONFIG["export_dir"]) / "test_output_pred_int64.bin")
test_y[:sample_count].numpy().astype(np.int64).tofile(Path(CONFIG["export_dir"]) / "test_labels_int64.bin")

with open(Path(CONFIG["export_dir"]) / "test_shapes.json", "w", encoding="utf-8") as f:
    json.dump({
        "sample_count": sample_count,
        "input_dim": int(sample_x.shape[1]),
        "num_classes": int(sample_logits.shape[1]),
    }, f, indent=2)

print("Exported correctness-check files.")

Exported correctness-check files.


In [19]:
# Export the full CIFAR-10 test embedding set for C++ inference
all_test_x = test_x.contiguous().float()
with torch.no_grad():
    all_test_logits = mlp(all_test_x.to(device)).cpu().contiguous().float()
    all_test_pred = torch.argmax(all_test_logits, dim=1)

export_dir = Path(CONFIG["export_dir"])

all_test_x.numpy().tofile(export_dir / "test_input_embeddings_all.bin")
all_test_logits.numpy().tofile(export_dir / "test_output_logits_all.bin")
all_test_pred.numpy().astype(np.int64).tofile(export_dir / "test_output_pred_int64_all.bin")
test_y.numpy().astype(np.int64).tofile(export_dir / "test_labels_int64_all.bin")

with open(export_dir / "test_shapes_all.json", "w", encoding="utf-8") as f:
    json.dump({
        "sample_count": int(all_test_x.shape[0]),
        "input_dim": int(all_test_x.shape[1]),
        "num_classes": int(all_test_logits.shape[1]),
    }, f, indent=2)

print(f"Exported full test set: {all_test_x.shape[0]} samples")

Exported full test set: 10000 samples


## Optional: quick prediction demo


In [18]:
@torch.no_grad()
def predict_from_embedding(model, x):
    model.eval()
    logits = model(x.to(device))
    pred = torch.argmax(logits, dim=1).cpu()
    return pred

pred = predict_from_embedding(mlp, test_x[:8])
for i, p in enumerate(pred.tolist()):
    print(f"sample {i}: pred={class_names[p]:>10s} | true={class_names[test_y[i].item()]:>10s}")

sample 0: pred=       cat | true=       cat
sample 1: pred=      ship | true=      ship
sample 2: pred=      ship | true=      ship
sample 3: pred=      bird | true=  airplane
sample 4: pred=      frog | true=      frog
sample 5: pred=      frog | true=      frog
sample 6: pred=automobile | true=automobile
sample 7: pred=      frog | true=      frog


## Suggested C++ inference shape

If you use this exported classifier in C++, the network will be:

- input: **512** (ResNet-18 embedding)
- hidden: **4096 -> 4096 -> 4096**
- output: **10**

That is a very clean workload for:
- serial baseline
- OpenMP parallelization
- MPI data-parallel batching
- MPI+OpenMP hybrid experiments
